# RetainIQ — Risk Classification (Critical / High / Medium / Low)

CLAUDE.md Sec 7 fixes four named churn-risk tiers: **Critical > 70%, High 50–70%, Medium 30–50%, Low < 30%**. `src/models/scoring.py` (Phase 2 + the `09-probability-scoring.md` bridge) already turns a customer's raw attributes into a calibrated churn probability, but nothing in the repo turns that probability into one of these four named tiers yet — `src/recommend/` is still empty. This notebook demonstrates `src/recommend/risk_tiers.py`, which fills that gap: a scalar classifier, a batch DataFrame classifier, a summary table, a distribution chart, and a convenience function that goes straight from raw customer attributes to `(churn_probability, risk_tier)`.

**Scope note:** this is the risk-tier half of CLAUDE.md Sec 14 Phase 4 only — no Next-Best-Action engine, no LLM insights, no FastAPI endpoint, no Streamlit view. See `.claude/specs/10-risk-classification.md` for the full spec, including the boundary-inclusivity interpretation resolved below.

In [1]:
import sys
from pathlib import Path

# Allow `import src...` when the notebook is run from notebooks/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.data.load_data import load_raw_data, TARGET_COLUMN
from src.models import scoring
from src.recommend import risk_tiers

pd.set_option("display.max_columns", None)
raw = load_raw_data()
print(f"{len(raw)} raw customers loaded")

7043 raw customers loaded


## 1. The four tiers, and a boundary-inclusivity note

CLAUDE.md's text — "Critical > 70%, High 50–70%, Medium 30–50%, Low < 30%" — states the 70% and 50% endpoints in two adjacent ranges each. Since Critical and Low are the two ranges given a *strict* inequality (`> 70%`, `< 30%`), a customer scored exactly 70% cannot be Critical (falls to High) and a customer scored exactly 30% cannot be Low (falls to Medium). This forces four disjoint, gap-free ranges:

- **Low** = [0%, 30%)
- **Medium** = [30%, 50%)
- **High** = [50%, 70%]
- **Critical** = (70%, 100%]

`risk_tiers.classify_risk_tier` implements exactly this rule; `risk_tiers.assign_risk_tiers` is a vectorized version tested to agree with it row-for-row.

In [2]:
print("MEDIUM_THRESHOLD:", risk_tiers.MEDIUM_THRESHOLD)
print("HIGH_THRESHOLD:   ", risk_tiers.HIGH_THRESHOLD)
print("CRITICAL_THRESHOLD:", risk_tiers.CRITICAL_THRESHOLD)
for p in [0.0, 0.29, 0.30, 0.49, 0.50, 0.69, 0.70, 0.71, 1.0]:
    print(f"  p={p:.2f} -> {risk_tiers.classify_risk_tier(p)}")

MEDIUM_THRESHOLD: 0.3
HIGH_THRESHOLD:    0.5
CRITICAL_THRESHOLD: 0.7
  p=0.00 -> Low
  p=0.29 -> Low
  p=0.30 -> Medium
  p=0.49 -> Medium
  p=0.50 -> High
  p=0.69 -> High
  p=0.70 -> High
  p=0.71 -> Critical
  p=1.00 -> Critical


## 2. Score every customer, then assign a tier

`scoring.score_customers` (from the `09-probability-scoring.md` bridge spec) takes raw customer rows and returns a calibrated churn probability. `risk_tiers.assign_risk_tiers` adds the `risk_tier` column on top.

In [3]:
features = raw.drop(columns=[TARGET_COLUMN])
scored = scoring.score_customers(features)
tiered = risk_tiers.assign_risk_tiers(scored)
tiered.head()

,customerID,churn_probability,churn_probability_pct,risk_tier
0,7590-VHVEG,0.6004,60.0,High
1,5575-GNVDE,0.0195,1.9,Low
2,3668-QPYBK,0.3052,30.5,Medium
3,7795-CFOCW,0.0199,2.0,Low
4,9237-HQITU,0.7094,70.9,Critical


## 3. Tier distribution

`risk_tier_summary` reports counts and percentages per tier, most-severe first — never a single aggregate number.

In [4]:
summary = risk_tiers.risk_tier_summary(scored)
summary

,risk_tier,count,pct
0,Critical,479,6.80
1,High,1055,14.98
2,Medium,1063,15.09
3,Low,4446,63.13


In [5]:
fig = risk_tiers.plot_risk_tier_distribution(scored)
fig.show()

## 4. Sanity check: does risk actually track observed churn?

CLAUDE.md Sec 9's "if this doesn't show up, something is wrong" spirit, applied to tiers instead of SHAP: the *observed* `Churn` rate per tier (using the real label, never fed into the model or the tier assignment itself) should increase strictly from Low to Critical. If it didn't, the calibrated probability the tiers are built on wouldn't be behaving as a real risk ladder.

In [6]:
labeled = tiered.copy()
labeled[TARGET_COLUMN] = raw[TARGET_COLUMN].map({"Yes": 1, "No": 0})  # aligns on index, same as tiered/scored

observed_churn_rate = (
    labeled.groupby("risk_tier", observed=False)[TARGET_COLUMN].mean() * 100
).reindex(risk_tiers.RISK_TIER_LABELS)
print(observed_churn_rate.round(1))

rates = observed_churn_rate.to_numpy()
is_increasing = all(rates[i] < rates[i + 1] for i in range(len(rates) - 1))
print(f"\nStrictly increasing Low->Critical: {is_increasing}")

risk_tier
Low          9.1
Medium      39.6
High        60.9
Critical    83.5
Name: Churn, dtype: float64

Strictly increasing Low->Critical: True


## 5. Example: raw customer straight to probability + tier

`classify_scored_customers` is the full raw-customer-in, `(churn_probability, risk_tier)`-out convenience path — the function a future Phase 5 `/predict` endpoint calls directly.

In [7]:
sample_customers = raw.head(5).drop(columns=[TARGET_COLUMN])
risk_tiers.classify_scored_customers(sample_customers)[
    ["customerID", "churn_probability", "churn_probability_pct", "risk_tier"]
]

,customerID,churn_probability,churn_probability_pct,risk_tier
0,7590-VHVEG,0.6004,60.0,High
1,5575-GNVDE,0.0195,1.9,Low
2,3668-QPYBK,0.3052,30.5,Medium
3,7795-CFOCW,0.0199,2.0,Low
4,9237-HQITU,0.7094,70.9,Critical


## Key findings

- **The four CLAUDE.md tiers are fully disjoint once the boundary-inclusivity rule is applied** — Critical strictly `> 70%`, Low strictly `< 30%`, with High/Medium filling the inclusive-lower gaps in between.
- **On the real dataset, the tier distribution is Low 63.1%, Medium 15.1%, High 15.0%, Critical 6.8%** (7,043 customers) — most customers are low-risk, with a meaningful ~22% in High+Critical worth a retention manager's attention.
- **Observed actual churn rate rises strictly with tier** (Low ~9%, Medium ~40%, High ~61%, Critical ~84%, verified above) — confirming the calibrated probability the tiers are built on is behaving as a real risk ladder, not an arbitrary cut.
- **`risk_tier` is a derived output, never a model input** — it's computed purely from `churn_probability` (itself already a model output) and never flows back into `src/features/preprocessing.py` or `src/models/train.py`.
- **`classify_scored_customers` is the function a future Phase 5 `/predict` endpoint will call directly** — no route or dashboard view exists yet; this notebook demonstrates the classification contract in isolation, matching the `08`/`09` precedent of shipping a capability before the endpoint that will expose it.